In [40]:
# Load & Inspect Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

df = pd.read_csv('../data/raw/cs-training.csv', index_col=0)
print(df.shape)
df.head()

count    1.202690e+05
mean     6.670221e+03
std      1.438467e+04
min      0.000000e+00
25%      3.400000e+03
50%      5.400000e+03
75%      8.249000e+03
max      3.008750e+06
Name: MonthlyIncome, dtype: float64 describe
1     9120.0
2     2600.0
3     3042.0
4     3300.0
5    63588.0
Name: MonthlyIncome, dtype: float64 Head
3008750.0 max
0.0 Min


In [41]:
#Exploratory Data Analysis (EDA)

In [70]:
# Data Cleaning
df_clean = df.copy()

#drop the row with age = 0 (impossible value, data entry error)
df_clean = df_clean[df_clean['age'] > 0]

#Fill missing MonthlyIncome with the median income
median_income = df_clean['MonthlyIncome'].median()
df_clean['MonthlyIncome'] = df_clean['MonthlyIncome'].fillna(median_income)

#Flag which rows originally had missing income
# (using the original df, since df_clean's nulls are already filled)
df_clean['income_was_missing'] = df['MonthlyIncome'].isnull().astype(int)

# DebtRatio was corrupted for rows where income was originally missing
# (debt / near-zero income = huge broken ratio)
# Fix: replace DebtRatio for those rows with the median DebtRatio
# calculated only from rows where income was reliable
reliable_debt_ratio_median = df_clean.loc[df_clean['income_was_missing'] == 0, 'DebtRatio'].median()
df_clean.loc[df_clean['income_was_missing'] == 1, 'DebtRatio'] = reliable_debt_ratio_median

# Cap remaining extreme DebtRatio values (genuine outliers, not missing-income related)
# using the 99th percentile of the reliable rows
debt_ratio_99th = df_clean.loc[df_clean['income_was_missing'] == 0, 'DebtRatio'].quantile(0.99)
df_clean['DebtRatio'] = df_clean['DebtRatio'].clip(upper=debt_ratio_99th)

# Cap RevolvingUtilizationOfUnsecuredLines outliers at the 99th percentile
# (this column's outliers were unrelated to missing income - just bad data)
util_99th = df_clean['RevolvingUtilizationOfUnsecuredLines'].quantile(0.99)
df_clean['RevolvingUtilizationOfUnsecuredLines'] = df_clean['RevolvingUtilizationOfUnsecuredLines'].clip(upper=util_99th)

# Step 7: Fill missing NumberOfDependents with 0 (both median and mode = 0)
df_clean['NumberOfDependents'] = df_clean['NumberOfDependents'].fillna(0)

# Final check: confirm no missing values remain anywhere
print(df_clean.isnull().sum())

SeriousDlqin2yrs                        0
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
income_was_missing                      0
dtype: int64


In [43]:
#Feature Engineering

In [44]:
#Train/Test Split

In [45]:
#Baseline Model — Logistic Regression

In [46]:
#Handle Class Imbalance (class weighting + SMOTE)

In [47]:
#Stronger Model — XGBoost

In [48]:
#Evaluation — compare all models

In [49]:
#Threshold Tuning — tie back to business problem

In [50]:
#Feature Importance / SHAP

In [51]:
#Save Model for Deployment